# RAWR walkthrough notebook

Step-by-step interactive version of `make all`. Run each cell after `pip install -e .`.
Useful for quickly iterating on a single problem (e.g. inspecting one RAWR case end-to-end).

In [ ]:
import json, yaml, pandas as pd
cfg = yaml.safe_load(open('../configs/default.yaml'))
from rawr.benchmark import SYNTHETIC, build_triplet_records
records = build_triplet_records(SYNTHETIC[:5], cfg['benchmark']['hint_types'], cfg['benchmark']['conditions'])
records[0]['variants']['misleading']['sycophancy']['prompt']

In [ ]:
from transformer_lens import HookedTransformer
model = HookedTransformer.from_pretrained('gpt2-small')
tokens = model.to_tokens(records[0]['variants']['clean']['prompt'])
_, cache = model.run_with_cache(tokens, names_filter=['blocks.6.hook_resid_post'])
cache['blocks.6.hook_resid_post'].shape

In [ ]:
from rawr.patching import patch_clean_into_misleading
out = patch_clean_into_misleading(
    model,
    clean_prompt=records[0]['variants']['clean']['prompt'],
    misleading_prompt=records[0]['variants']['misleading']['sycophancy']['prompt'],
    layer=6, max_new_tokens=64,
)
out